# 02 - Data Exploration

This notebook explores the dataset using normal imports from the project code. It imports `data_loader.py` directly instead of running inline Python through `subprocess`.

## Step 1 - Setup

In [ ]:
from pathlib import Path
import csv
import json
import sys

def find_notebooks_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd / "salient-object-detection-cnn" / "notebooks"]
    for parent in cwd.parents:
        candidates.extend([parent / "notebooks", parent / "salient-object-detection-cnn" / "notebooks"])
    for candidate in candidates:
        if (candidate / "notebook_utils.py").exists():
            return candidate
    raise FileNotFoundError("Could not find notebooks/notebook_utils.py")

NOTEBOOKS_DIR = find_notebooks_dir()
sys.path.insert(0, str(NOTEBOOKS_DIR))

from notebook_utils import PROJECT_DIR

PROJECT_DATA_DIR = PROJECT_DIR / "data"
if not PROJECT_DATA_DIR.exists():
    PROJECT_DATA_DIR = (PROJECT_DIR / "../data").resolve()

sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project directory: {PROJECT_DIR}")

## Step 2 - Inspect Metadata And Manifest

In [ ]:
metadata_path = PROJECT_DIR / "data/metadata.csv"
print("metadata.csv exists:", metadata_path.exists())

if metadata_path.exists():
    with metadata_path.open(newline="") as file:
        metadata_rows = list(csv.DictReader(file))
    print("metadata rows:", len(metadata_rows))
    print("first row:", metadata_rows[0] if metadata_rows else "empty")

manifest_path = PROJECT_DIR / "pre-processed/manifest.json"
print("manifest exists:", manifest_path.exists())
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("manifest counts:", manifest.get("counts"))
else:
    print("Run 01_data_preprocessing.ipynb before exploring preprocessed tensors.")

## Step 3 - Load Preprocessed Splits Through `data_loader.py`

In [ ]:
import importlib
import data_loader

importlib.reload(data_loader)
from data_loader import create_datasets

train_dataset, val_dataset, test_dataset = create_datasets(
    data_dir=str(PROJECT_DIR / "pre-processed"),
    image_size=128,
    use_preprocessed=True,
 )

for split_name, dataset in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    sample = dataset[0]
    print(f"{split_name} samples: {len(dataset)}")
    print(f"{split_name} image shape: {tuple(sample['image'].shape)}")
    print(f"{split_name} mask shape: {tuple(sample['mask'].shape)}")
    print(f"{split_name} image range: {sample['image'].min().item():.4f} to {sample['image'].max().item():.4f}")
    print(f"{split_name} mask range: {sample['mask'].min().item():.4f} to {sample['mask'].max().item():.4f}")
    print(f"{split_name} source image: {sample['image_path']}")
    print("-" * 60)

## Step 4 - Check Raw Loader Fallback

In [ ]:
from data_loader import DUTSDataset

raw_train = DUTSDataset(data_dir=str(PROJECT_DATA_DIR), split="train", image_size=128, augment=False)
raw_val = DUTSDataset(data_dir=str(PROJECT_DATA_DIR), split="val", image_size=128, augment=False)
raw_test = DUTSDataset(data_dir=str(PROJECT_DATA_DIR), split="test", image_size=128, augment=False)

print("raw train samples:", len(raw_train))
print("raw val samples:", len(raw_val))
print("raw test samples:", len(raw_test))
sample = raw_train[0]
print("raw sample image shape:", tuple(sample["image"].shape))
print("raw sample mask shape:", tuple(sample["mask"].shape))